````markdown
# Quantum Graph Neural Networks (QGNNs) — Detailed Notes (Session 17)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller  

> **Purpose.** Turn the slide bullets into a stand-alone reference for designing, training, and evaluating **hybrid QGNNs**. We cover graph→quantum encodings, quantum message passing blocks, scalable subgraphing (ego-graphs), Qiskit+PyTorch patterns, and practical pitfalls. Mini-exercises (with brief answers) appear at the end.

---

## Session road-map
1. Recap: QCNNs and why graphs are different  
2. What is a QGNN? (message passing with PQCs)  
3. Graph **encoding** into qubits (angle maps, ego-graphs, padding)  
4. **Quantum message passing** blocks (shared parameters, entanglers)  
5. Hybrid architectures (quantum block + classical aggregator)  
6. Qiskit ML implementation patterns (EstimatorQNN + Torch)  
7. Scalability and complexity accounting  
8. Practical tips, pitfalls, and lab outline

---

## 0) Recap — from QCNNs to QGNNs
- QCNNs exploit **locality + parameter sharing** on grids (images).  
- Graphs are **irregular**: variable degree, no fixed spatial ordering, permutation symmetries.  
- QGNNs carry over the ideas (local receptive fields, shared filters), but operate on **neighbourhoods** instead of fixed patches.

---

## 1) What is a QGNN?
A **hybrid** model that replaces some GNN layers with **parameterised quantum circuits (PQCs)** to process local subgraphs and produce embeddings.

**Message–passing intuition** (one layer):
1. **Gather** a node’s neighbourhood (e.g., radius-1 ego graph).  
2. **Encode** node & neighbour features onto qubits.  
3. **Interact** via a PQC with **shared parameters** across all nodes.  
4. **Read out** expectations → a node embedding (or an edge/graph embedding).  
5. **Aggregate** across nodes/edges (classically) and repeat for L layers.

**Why bother?** Entanglement allows non-classical interactions among neighbour features with **low parameter count**, and the hybrid loop keeps depth shallow (NISQ-friendly).

---

## 2) Encoding graphs into qubits

### 2.1 Neighbourhood (ego-graph) decomposition
- Radius-1 “ego” around node $v$: center $v$ + up to $K-1$ neighbours → **K qubits**.  
- For degree $>\!K-1$: pick top-K by score (degree, centrality) or random sample; record a **mask**.

### 2.2 Feature → angle encoding (common & hardware-friendly)
- For scalar feature $x$: $R_Y(\alpha x)$ (or $R_Z$); normalise $x \in [-1,1]$ → angles in $[-\pi,\pi]$.  
- For multi-dim features:  
  - **Data re-uploading:** alternate $U_\phi(x^{(1)}) \to U(\theta)\to U_\phi(x^{(2)}) …$  
  - Or **compress** classically (PCA/MLP) to 1–2 scalars per node.

### 2.3 Ordering & permutation issues
- Choose a **deterministic order** for neighbours (e.g., sort by degree, id).  
- Compensate with **symmetric pooling** downstream (mean/max), so small order changes do not hurt.

---

## 3) Quantum message-passing block (PQC)

### 3.1 Filter circuit $U(\boldsymbol{\theta})$ on K qubits
- **Local rotations:** $R_Y, R_Z$ with shared parameters across all ego-graphs.  
- **Entanglers:** linear/star CX (center→neighbour), or ZZ-couplings $e^{-i\gamma Z_i Z_j}$ along ego edges.  
- **Depth:** 1–2 reps (NISQ).  
- **Parameter sharing:** same $\boldsymbol{\theta}$ for every node at a given layer (like a GNN layer’s weights).

### 3.2 Readout (node embedding)
- Measure expectations on selected qubits, e.g.,
  \[
  h_v = \big[\,\langle Z_{0}\rangle,\; \tfrac{1}{K-1}\sum_{j=1}^{K-1}\langle Z_j\rangle \,\big]
  \]
- Optionally include cross-terms (e.g., $\langle Z_0Z_j\rangle$) via additional circuits.

### 3.3 Stacking layers
- Repeat: $h_v^{(l+1)} = \text{ClassicalAgg}\big(h_v^{(l)},\{h_u^{(l)}: u\in N(v)\}\big)$ or plug $h^{(l)}$ back into the next quantum block.

---

## 4) Hybrid QGNN architectures

**Node-level tasks (classification/regression)**
- Quantum block → node embeddings $h_v$ → classical message aggregation (mean/max/attention) → MLP → loss.

**Graph-level tasks (molecules)**
- Quantum block per node or per **edge/fragment** → node/edge embeddings → **global pooling** (sum/mean) → MLP → loss.

**Edge-level tasks (link prediction)**
- Build a joint ego-graph around $(u,v)$ → quantum block → edge score.

> **Rule of thumb:** keep the quantum part **small + repeated**, and delegate variable-size aggregation to classical code.

---

## 5) Qiskit ML implementation (EstimatorQNN + PyTorch)

> Prefer **primitives** (`Estimator`, `Sampler`) with **EstimatorQNN**/**SamplerQNN** and `TorchConnector`. (The legacy `CircuitQNN`/Opflow APIs are deprecated.)

### 5.1 One PQC filter on a K-qubit ego-graph
```python
# pip install qiskit qiskit-aer qiskit-machine-learning torch networkx
import networkx as nx
import numpy as np, torch
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import TwoLocal
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

K = 5  # center + up to 4 neighbors
# Data angles per qubit (one scalar per node here; re-upload for multi-dim)
phi = ParameterVector('x', length=K)

# 1) Encoding: angle-encode K scalars
enc = QuantumCircuit(K)
for q in range(K):
    enc.ry(phi[q], q)

# 2) Shared filter with light entanglement (center=qubit 0)
ans = TwoLocal(K, rotation_blocks=['ry','rz'], entanglement_blocks='cx',
               entanglement=[(0,j) for j in range(1,K)], reps=1)

qc = enc.compose(ans)

# 3) Expectation readout on center qubit (can add others)
qnn = EstimatorQNN(circuit=qc,
                   input_params=list(phi),
                   weight_params=list(ans.parameters),
                   estimator=Estimator(shots=1024))

filter_layer = TorchConnector(qnn)  # torch.nn.Module: forward(x: [K]) -> scalar
```

### 5.2 Slide the filter across ego-graphs, aggregate, predict
```python
def ego_pack(G:nx.Graph, v:int, K:int):
    # Build radius-1 ego-graph; select up to K-1 neighbors with deterministic rule
    nbrs = sorted(list(G.neighbors(v)))[:K-1]
    nodes = [v] + nbrs
    # Extract one scalar feature per node; pad with zeros if <K
    x = [G.nodes[u].get('feat', 0.0) for u in nodes] + [0.0]*max(0, K-len(nodes))
    return np.array(x, dtype=np.float32)

class QGNNHead(torch.nn.Module):
    def __init__(self, filter_mod, K, out_dim=1):
        super().__init__()
        self.filter = filter_mod
        self.K = K
        self.readout = torch.nn.Sequential(
            torch.nn.Linear(1, 16), torch.nn.ReLU(),
            torch.nn.Linear(16, out_dim)
        )
    def forward(self, batch_graphs):
        # batch_graphs: list of (G, label) with node feats pre-attached
        graph_embeds = []
        for G,_ in batch_graphs:
            node_vals = []
            for v in G.nodes():
                x = torch.tensor(ego_pack(G, v, self.K))
                z = self.filter(x)                 # scalar node embedding
                node_vals.append(z.squeeze())
            H = torch.stack(node_vals)             # [|V|]
            g = H.mean()                            # simple global mean pooling
            graph_embeds.append(g.unsqueeze(0))     # shape [1]
        Hgraph = torch.stack(graph_embeds)          # [B,1]
        return self.readout(Hgraph).squeeze(-1)
```

> **Notes.**  
> • For multi-dim node features, either **compress** first (e.g., MLP→1D) or **re-upload** angles.  
> • For node classification, replace global mean with per-node heads.  
> • To vectorise, batch multiple `input_data` into one Estimator call to amortise latency.

---

## 6) Training loop & losses
- **Binary graph classification:** BCEWithLogits on graph scores.  
- **Multiclass:** Cross-entropy on graph logits.  
- **Node classification:** loss over nodes with mask for labeled subset.

**Optimisers**
- *Gradient-based:* Adam using parameter-shift (EstimatorQNN handles this under the hood).  
- *Gradient-free:* SPSA (robust to shot noise; good for small parameter counts).

**Batching**
- Build mini-batches of graphs; within each graph, evaluate **all ego-patches** for a given $\theta$ before updating.

---

## 7) Scalability & complexity

Let:
- $B$ = batch size (graphs/step), $|V|_{\text{avg}}$ = avg #nodes/graph,  
- $K$ = ego size (qubits), $P$ = #trainable parameters in the filter,  
- $S$ = shots, $L$ = #QGNN layers (quantum).

**Per step cost (parameter-shift, expectation readout)** ≈  
\[
\underbrace{B |V|_{\text{avg}}}_{\text{ego patches}}
\times\underbrace{(2P)}_{\text{grad evaluations}}
\times\underbrace{S}_{\text{shots}}
\]
(ignoring constant overhead). Use **mini-batches** and cache transpiled circuits.

**Memory/width bottleneck:** fixed K bounds max degree handled directly; for higher degrees, sample or pool neighbours classically.

**Depth/noise:** keep reps ≤ 1–2 per layer; prefer center–neighbour entanglers aligned with hardware topology.

---

## 8) Practical guidance & pitfalls

- **Permutation invariance:** pick a deterministic neighbour order *and* use symmetric pooling (mean/attention).  
- **Variable degree:** pad with zeros + a **mask**; or sample fixed-size neighbour sets (consistent seed).  
- **Feature scaling:** normalise to $[-1,1]$; otherwise rotations saturate → flat gradients.  
- **Shot budgeting:** start 256–512 for exploration; increase (1–4k) near convergence; average across batches.  
- **Readout calibration:** invert confusion matrix to debias $\langle Z\rangle$.  
- **Overfitting (small datasets):** L2 on $\boldsymbol{\theta}$, early stopping, data augmentation on graphs (edge drop with care).  
- **Latency:** batch many ego patches per $\theta$; bind parameters without re-transpiling.

---

## 9) Applications & patterns

- **Molecules (graph-level):** toxicity/activity (e.g., MUTAG-like). Use atom valence/charge as features; K=5–6.  
- **Social/link tasks (edge-level):** score candidate edges via joint ego-graph around $(u,v)$.  
- **Heterogeneous graphs:** use **type-specific** encoders classically, then a shared quantum block.

**When it helps (today)**  
Small graphs, strong local chemistry/structure, few-shot regimes, or as a **feature extractor** inside a classical GNN/CNN pipeline.

---

## 10) Mini-exercises (answers in Appendix)
1. **Ego size vs degree.** Your graphs have max degree 12, but K=6 qubits. Propose two deterministic neighbour-selection rules and discuss bias implications.  
2. **Permutation robustness.** Show that mean pooling over node embeddings makes the graph representation invariant to node relabeling. What invariances are *not* handled by mean pooling?  
3. **Complexity accounting.** For $B=8$, $|V|_{\text{avg}}=30$, $P=20$, $S=1024$, estimate circuit evaluations per gradient step (parameter-shift).  
4. **Noise estimate.** Two-qubit error 1% and 8 entanglers per ego ⇒ rough multiplicative shrink on expectation magnitude? Suggest two mitigations.  
5. **Kernel fallback.** If gradients vanish, show how to replace the PQC block with a **quantum kernel** on ego-graphs while keeping the same encoding.

---

## 11) Summary (Session 17)
- QGNNs process **local neighbourhoods** with a **shared** PQC filter, read out expectations, and aggregate classically.  
- Use **ego-graphs** to fit fixed qubit budgets; encode with angle maps and re-upload if needed.  
- Implement with **EstimatorQNN + TorchConnector**; batch ego patches; calibrate readout; keep circuits shallow.  
- Evaluate against matched classical GNN baselines and report **shot/latency** costs.  
- Best used as **hybrid components** on NISQ-era devices.

---

## 12) Looking ahead
- **Next Session:** Final Project — proposal guidelines, milestones, and rubric (choose from QCNN/QGNN/VQE/QAOA tracks).  
- **Homework 4 (QGNN):**  
  1) Build a K=5 QGNN head for a small molecular dataset; report accuracy vs shots.  
  2) Compare re-uploading vs PCA compression for 4-dim node features.  
  3) (Bonus) Swap in a quantum kernel on ego-graphs and compare to the variational block.

---

## Appendix — mini-exercise solutions (sketch)

1. **Neighbour selection:** (i) highest-degree first (captures influential nodes) (ii) deterministic hash/order by node id (unbiased wrt structure). Bias: (i) favours hubs; (ii) ignores structural salience. Mitigate by alternating rules across epochs.  
2. **Permutation invariance:** If $H=\{h_{\pi(v)}\}$ is a permutation of $H$, then $\text{mean}(H)$ is unchanged; mean is invariant to node relabelings. Not handled: changes that alter neighbourhood content (subgraph isomorphisms) or require edge-order invariance within the ego encoding itself.  
3. **Evaluations/step:** $B|V|_{\text{avg}}\times 2P = 8\times 30\times 40 = 9{,}600$ expectation evaluations (each with 1024 shots).  
4. **Shrink:** $(1-0.01)^8 \approx 0.92$. Mitigate with (a) topology-aware routing to reduce two-qubit count/SWAPs; (b) zero-noise extrapolation + dynamical decoupling; also consider fewer entanglers.  
5. **Kernel swap:** Keep the same encoding $U_\phi(\text{ego})$; compute $K_{ij}=|\langle 0|U_\phi^\dagger(\text{ego}_i)U_\phi(\text{ego}_j)|0\rangle|^2$; train an SVM or kernel ridge on pooled ego-kernels per graph. Removes variational parameters → no barren plateaus, but costs $O(N^2)$ kernel evals.
````
